# Tutorial 14 LLM-Based Agents: Tools, Recovery, and Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_14_LLM_based_Agent/tutorial_14_pytorch.ipynb)

Build a small chemistry assistant using a real pretrained language model running with PyTorch **inside Colab**. The LLM selects a descriptor; RDKit performs the calculation; a bounded controller handles name resolution and records evidence.

**Start:** Open in Colab → connect → **Runtime → Run all**. Dependencies and model downloads are automatic. No API key, paid inference service, Drive mount, or previous notebook state is required. Private repository access requires GitHub authorization in Colab, or download this notebook and use **File → Upload notebook**.

**Prerequisites:** Python functions/dictionaries, PyTorch inference, and basic train/validation/test separation. No chemistry background is assumed.

**Resources:** A roughly 1 GB public model download; CPU inference uses float32 (about 2 GB for weights, plus runtime overhead). Expect several minutes, depending on downloads and CPU speed. GPU is optional. This lesson performs inference, not model training.

## Goals and scope

1. Compare a generated answer with a deterministic chemical descriptor.
2. Inspect tokenizer inputs and real LLM action scores.
3. Build a tool registry with validated JSON requests, observations, and explicit failures.
4. Trace a bounded action–observation workflow, including name-resolution recovery.
5. Select between prompts using a small development set, then evaluate held-out wording.
6. Export auditable traces and verify reloads without duplicating large weights.

**Changes from upstream:** The paid GPT/LiteLLM calls are replaced by public Qwen2.5-0.5B-Instruct inference. The live name resolver becomes a small explicit local catalogue. The expensive DSPy/GSM8K search becomes a transparent two-prompt selection exercise on tool routing. This is **not a DSPy implementation or reproduction of GSM8K results**. MRKL-style modularity and action–observation ideas remain; the controller is deliberately constrained, not an unrestricted autonomous laboratory agent.


## 1. Install and configure

Retain Colab's PyTorch. Install missing packages only. [Qwen's model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) documents the Transformers interface; the checkpoint revision is pinned below. We use safetensors and do not execute repository-supplied model code. The model's Apache-2.0 license is separate from the tutorial license.

`QUICK_RUN=True` limits generated answer length. `False` increases that length; it does not enlarge the evaluation dataset. The scored routing calls still use the actual neural model in either mode.


In [ ]:
import importlib.util, subprocess, sys
packages={"torch":"torch", "transformers":"transformers>=4.45,<6", "rdkit":"rdkit",
          "numpy":"numpy", "pandas":"pandas", "matplotlib":"matplotlib", "safetensors":"safetensors"}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])


In [ ]:
import json, hashlib, platform, random, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdMolDescriptors, Draw
from IPython.display import display
%matplotlib inline
SEED=42;QUICK_RUN=True;DEVICE_PREFERENCE="auto"
assert DEVICE_PREFERENCE in {"auto","cpu"}
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
device=torch.device("cuda" if DEVICE_PREFERENCE=="auto" and torch.cuda.is_available() else "cpu")
OUT=Path("outputs/tutorial_14");OUT.mkdir(parents=True,exist_ok=True)
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION="7ae557604adf67be50417f59c2c2f167def9a775"
versions={"python":platform.python_version(),"torch":str(torch.__version__),"transformers":transformers.__version__,
          "rdkit":rdBase.rdkitVersion,"numpy":np.__version__,"pandas":pd.__version__,"matplotlib":matplotlib.__version__,
          "device":str(device),"quick_run":QUICK_RUN,"model_id":MODEL_ID,"model_revision":MODEL_REVISION}
display(pd.Series(versions));started=time.perf_counter();checks={};figures=[]
def finish(fig,name):
    fig.tight_layout();fig.savefig(OUT/name,bbox_inches="tight");figures.append(name);plt.show()
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION,trust_remote_code=False)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,trust_remote_code=False,
                                         use_safetensors=True,torch_dtype=torch.float32).to(device).eval()
print("Loaded parameters:",sum(p.numel() for p in model.parameters()))
def chat_inputs(messages):
    text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    return tokenizer(text,return_tensors="pt").to(device)
def generate(messages,max_new_tokens=48):
    inputs=chat_inputs(messages)
    with torch.inference_mode():out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0,inputs.input_ids.shape[1]:],skip_special_tokens=True)


## 2. Tools return evidence, not guesses

A **SMILES** string encodes molecular structure. Hydrogen-bond donors (HBD) and acceptors (HBA) are RDKit-defined counts. `MolWt` is average molecular weight in g/mol, not exact isotopic mass. Results depend on the supplied protonation/tautomer state and descriptor definitions; none predicts efficacy, toxicity, bulk material performance, or synthesis feasibility.

The name catalogue contains only six teaching examples. Unknown names return an error instead of inventing a structure. Molecule input is supplied separately from the question to keep entity extraction outside this small lesson's scope. Invalid input, unknown tools, and extra arguments are rejected before execution. No `eval`, shell tool, or dynamic Python execution is used.

**Descriptor convention:** The tested RDKit functions return HBD=0/HBA=0 for the standalone water SMILES `O`. This reflects these particular descriptor patterns, not a claim that water cannot form hydrogen bonds. Always state the algorithm and molecular representation when interpreting counts.


In [ ]:
CATALOGUE={"ethanol":"CCO","aspirin":"CC(=O)Oc1ccccc1C(=O)O","caffeine":"Cn1c(=O)c2c(ncn2C)n(C)c1=O",
           "benzene":"c1ccccc1","glycine":"NCC(=O)O","water":"O"}
DESCRIPTORS={"hbd":(rdMolDescriptors.CalcNumHBD,"count"),"hba":(rdMolDescriptors.CalcNumHBA,"count"),"mw":(Descriptors.MolWt,"g/mol")}
def molecule_from_smiles(text):
    if not isinstance(text,str) or not text.strip() or len(text)>300:raise ValueError("Expected a nonempty SMILES string of at most 300 characters")
    with rdBase.BlockLogs():mol=Chem.MolFromSmiles(text)
    if mol is None or mol.GetNumAtoms()==0:raise ValueError("Invalid SMILES; try resolve_name for a catalogue name")
    return mol
def dispatch(request):
    try:
        if not isinstance(request,dict) or set(request)!={"tool","arguments"}:raise ValueError("Expected exactly tool and arguments")
        tool=request["tool"];args=request["arguments"]
        if not isinstance(tool,str) or not isinstance(args,dict):raise ValueError("Invalid request types")
        key="name" if tool=="resolve_name" else "smiles"
        if set(args)!={key} or not isinstance(args[key],str):raise ValueError("Invalid arguments")
        if tool=="resolve_name":
            name=args["name"].strip().lower()
            if name not in CATALOGUE:raise ValueError("Name absent from teaching catalogue")
            return {"ok":True,"tool":tool,"name":name,"smiles":CATALOGUE[name],"source":"embedded teaching catalogue"}
        if tool not in DESCRIPTORS:raise ValueError("Unknown tool")
        mol=molecule_from_smiles(args["smiles"]);fn,unit=DESCRIPTORS[tool]
        return {"ok":True,"tool":tool,"smiles":Chem.MolToSmiles(mol),"value":float(fn(mol)),"unit":unit,"source":"RDKit "+rdBase.rdkitVersion}
    except (ValueError,TypeError,KeyError) as error:
        return {"ok":False,"error":str(error)}
def call(tool,value):return dispatch({"tool":tool,"arguments":{"name" if tool=="resolve_name" else "smiles":value}})
assert call("hbd","CCO")["value"]==1
assert call("hba","CCO")["value"]==1
assert abs(call("mw","O")["value"]-18.015)<0.01
assert not call("hbd","not a molecule")["ok"]
assert not call("delete_file","CCO")["ok"]
assert not dispatch({"tool":"hbd","arguments":{"smiles":"CCO","extra":1}})["ok"]
assert not call("resolve_name","unknownium")["ok"]
checks["tool_values_schema_and_errors"]=True
rows=[]
for name,smiles in CATALOGUE.items():rows.append({"name":name,**{tool:call(tool,smiles)["value"] for tool in DESCRIPTORS}})
display(pd.DataFrame(rows))
fig,axes=plt.subplots(2,3,figsize=(12,7))
for ax,(name,smiles) in zip(axes.flat,CATALOGUE.items()):
    ax.imshow(Draw.MolToImage(Chem.MolFromSmiles(smiles),size=(350,250)));ax.axis("off");ax.set_title(f"{name}: HBD={call('hbd',smiles)['value']:g}, HBA={call('hba',smiles)['value']:g}")
finish(fig,"01_molecules_and_tools.png")


## 3. A generated answer versus a calculated answer

Here the LLM receives SMILES and generates an answer with no tools. The next output is an independently computed descriptor. Read both even if they happen to agree: one example cannot establish reliability. We keep the raw response rather than silently replacing it with the tool result. Greedy decoding makes the generation repeatable within a fixed environment, not necessarily correct.


In [ ]:
direct_question=f"How many hydrogen bond donors are in the molecule with SMILES {CATALOGUE['aspirin']}? Reply with the number only."
direct_answer=generate([{"role":"user","content":direct_question}],max_new_tokens=24 if QUICK_RUN else 80)
direct_reference=call("hbd",CATALOGUE["aspirin"])
print("LLM without tools:",repr(direct_answer));print("RDKit evidence:",direct_reference)
# Strict scoring: explanations or multiple numbers are not treated as a valid number-only answer.
match=re.fullmatch(r"\s*(\d+(?:\.\d+)?)\s*",direct_answer)
direct_exact=bool(match and float(match.group(1))==direct_reference["value"])
print("Strict numeric match:",direct_exact)


## 4. Constrain the action space and inspect model scores

Instead of trusting arbitrary generated JSON, ask the model to select one of four single-token action labels. Take the next-token logits for these labels and select the largest. This is a real pretrained LLM forward pass, **not a keyword router or scripted response**. It guarantees an allowed label, not correct intent recognition.

The softmax shown below is conditional on four labels; it is **not calibrated confidence** and ignores all other vocabulary tokens. A narrow router can still choose the wrong tool or fail to reject an unsupported request. Final numerical values will always come from executed tools, but correct arithmetic alone cannot guarantee the answer addresses the question.

Two prompt candidates are fixed before evaluation. Few-shot demonstrations below are separate from development/test wording. We will select the prompt on development accuracy only. This illustrates the evaluate–select pattern behind prompt optimization without claiming to run DSPy or train model weights.


In [ ]:
LABELS={"A":"hbd","B":"hba","C":"mw","D":"unsupported"}
PROMPTS={
 "brief":"Choose the requested molecular property. A: hydrogen bond donor count. B: hydrogen bond acceptor count. C: molecular weight. D: any other request. Reply with one letter only.",
 "examples":"Classify the requested property, not the molecule. A = number of hydrogen bond donors (HBD). B = number of hydrogen bond acceptors (HBA). C = average molecular weight or molar mass. D = any unsupported property or multiple properties. Examples: 'Count HBD' -> A; 'Count HBA' -> B; 'What is the molar mass?' -> C; 'What color is it?' -> D. Return only A, B, C, or D."
}
label_ids=[tokenizer.encode(label,add_special_tokens=False) for label in LABELS]
assert all(len(ids)==1 for ids in label_ids);label_ids=[ids[0] for ids in label_ids]
route_cache={};model_route_calls=0
def route(question,prompt_name="brief"):
    global model_route_calls
    key=(question,prompt_name)
    if key not in route_cache:
        if not isinstance(question,str) or not 0<len(question)<=500:raise ValueError("Question length must be 1–500 characters")
        inputs=chat_inputs([{"role":"system","content":PROMPTS[prompt_name]},{"role":"user","content":question}])
        with torch.inference_mode():scores=model(**inputs).logits[0,-1,label_ids].float().cpu()
        probs=scores.softmax(0);label=list(LABELS)[int(scores.argmax())]
        route_cache[key]={"tool":LABELS[label],"label":label,"conditional_scores":dict(zip(LABELS,probs.tolist())),"input_tokens":int(inputs.input_ids.shape[1])}
        model_route_calls+=1
    return json.loads(json.dumps(route_cache[key]))
example_question="How many hydrogen bond acceptors does this molecule have?"
example_route=route(example_question);display(example_route)
inputs=chat_inputs([{"role":"system","content":PROMPTS["brief"]},{"role":"user","content":example_question}])
display(pd.DataFrame({"token_id":inputs.input_ids[0,-20:].cpu().tolist(),"token":tokenizer.convert_ids_to_tokens(inputs.input_ids[0,-20:].cpu().tolist())}))
fig,ax=plt.subplots(figsize=(7,4));ax.bar([LABELS[k] for k in LABELS],list(example_route["conditional_scores"].values()))
ax.set(ylim=(0,1),ylabel="Conditional score (not calibrated)",title="Real LLM next-token scores over allowed actions");finish(fig,"02_action_scores.png")
checks["single_token_allowed_actions"]=True


## 5. A bounded action–observation controller

The LLM selects the descriptor. The **controller**, not the LLM, then:

1. Attempts that descriptor on the supplied molecule text.
2. On invalid SMILES, resolves the unchanged text using the catalogue.
3. Retries the same descriptor on the resolved SMILES, then formats the observed result.

Every attempt counts against a three-tool-call budget. An unknown name or failed tool yields explicit abstention. The resolver cannot replace an unknown molecule with a convenient known one. Trace records contain actions and observations, not purported hidden reasoning.

This error recovery is a programmed policy. The upstream agent asked an LLM to repair its own calls; that more general behavior is not established here. Separating these responsibilities makes it possible to test controller correctness independently of model accuracy.


In [ ]:
def run_agent(question,molecule,prompt_name="brief",max_tool_calls=3,router=route):
    if not isinstance(molecule,str) or not 0<len(molecule)<=300:raise ValueError("Molecule must be a short name or SMILES string")
    if not isinstance(max_tool_calls,int) or not 1<=max_tool_calls<=3:raise ValueError("Tool budget must be 1–3")
    decision=router(question,prompt_name);trace=[];selected=decision.get("tool")
    result={"question":question,"input":molecule,"decision":decision,"trace":trace,"status":"abstained","answer":None,"evidence":None}
    if selected not in DESCRIPTORS:
        result["reason"]="Unsupported or invalid action";return result
    next_tool=selected;argument=molecule
    for step in range(1,max_tool_calls+1):
        request={"tool":next_tool,"arguments":{"name" if next_tool=="resolve_name" else "smiles":argument}}
        observation=dispatch(request);trace.append({"step":step,"request":request,"observation":observation})
        if observation["ok"] and next_tool==selected:
            result.update(status="answered",evidence=observation,answer=f"{selected.upper()} = {observation['value']:.3f} {observation['unit']} (RDKit; SMILES {observation['smiles']}).")
            return result
        if observation["ok"] and next_tool=="resolve_name":
            next_tool=selected;argument=observation["smiles"]
        elif next_tool==selected and step==1:
            next_tool="resolve_name";argument=molecule
        else:
            result["reason"]=observation.get("error","Tool failed");return result
    result["reason"]="Tool-call budget exhausted";return result
live_demo=run_agent("How many hydrogen bond donors does this molecule have?","aspirin")
display(live_demo)
fig,ax=plt.subplots(figsize=(12,3));ax.axis("off")
steps=live_demo["trace"]
for i,step in enumerate(steps):
    box_x=i*0.34
    obs=step["observation"];label=f"{step['step']}. {step['request']['tool']}\n"+("OK" if obs["ok"] else "Invalid input")
    ax.add_patch(FancyBboxPatch((box_x,0.25),0.27,0.5,boxstyle="round,pad=0.02",facecolor="#dcfce7" if obs["ok"] else "#fee2e2",edgecolor="#475569"))
    ax.text(box_x+0.135,0.5,label,ha="center",va="center",fontsize=11)
    if i<len(steps)-1:ax.annotate("",xy=(box_x+0.32,0.5),xytext=(box_x+0.28,0.5),arrowprops={"arrowstyle":"->"})
ax.set_title("Observed action–observation trace: LLM-selected descriptor, programmed recovery")
finish(fig,"03_agent_trace.png")


## 6. Fault injection: test the controller without conflating it with the LLM

The following routers are explicitly **test fixtures**. They isolate error handling from neural predictions and are never used for model accuracy evaluation. A fabricated observation is not accepted as evidence: the controller obtains observations only from its own dispatcher. The tests check bounded calls, no answer on unknown molecules, immutable input, malformed schemas, and answer/evidence consistency.


In [ ]:
def fixture_hbd(question,prompt_name):return {"tool":"hbd","fixture":True}
def fixture_bad(question,prompt_name):return {"tool":"shell","fixture":True}
repaired=run_agent("Fixture","aspirin",router=fixture_hbd)
assert repaired["status"]=="answered" and len(repaired["trace"])==3 and repaired["evidence"]["value"]==1
assert repaired["input"]=="aspirin" and not repaired["trace"][0]["observation"]["ok"]
unknown=run_agent("Fixture","unknownium",router=fixture_hbd);assert unknown["status"]=="abstained" and unknown["answer"] is None
budget=run_agent("Fixture","aspirin",max_tool_calls=1,router=fixture_hbd);assert budget["status"]=="abstained" and len(budget["trace"])==1
bad=run_agent("Fixture","CCO",router=fixture_bad);assert bad["status"]=="abstained" and not bad["trace"]
valid=run_agent("Fixture","CCO",router=fixture_hbd);assert valid["status"]=="answered" and len(valid["trace"])==1
assert not dispatch({"tool":"hbd","arguments":{"smiles":"CCO"},"observation":{"value":99}})["ok"]
checks["recovery_budget_unknown_action_and_evidence"]=True
print("PASS: six controller/schema fault checks. These are fixtures, not LLM accuracy results.")


## 7. Prompt selection, then held-out evaluation

The eight development questions and eight held-out questions are hand-written teaching cases, not a statistical benchmark. Each set includes donors, acceptors, weight, and unsupported requests. Exact wording is disjoint; semantic overlap is intentional. Development accuracy selects one of the two fixed prompts; ties prefer the first. Test labels are never passed into the router or controller.

Routing accuracy measures intent selection. End-to-end correctness also checks status, canonical structure, units, and numeric result. Unsupported requests are correct only when the system abstains. An `answered` status is a successful tool execution, not automatically a correct answer to the question.


In [ ]:
DEV=[
 ("Count the hydrogen bond donor groups.","ethanol","hbd"),
 ("Give the number of HBD sites.","aspirin","hbd"),
 ("Count the hydrogen bond acceptor sites.","caffeine","hba"),
 ("Give the HBA count.","water","hba"),
 ("Calculate the molecular weight in grams per mole.","glycine","mw"),
 ("What is the molar mass?","benzene","mw"),
 ("What is the boiling temperature?","ethanol","unsupported"),
 ("Predict the crystal band gap.","aspirin","unsupported")]
TEST=[
 ("How many groups can donate a hydrogen bond?","glycine","hbd"),
 ("Report the hydrogen-bond donor count for this structure.","water","hbd"),
 ("How many groups can accept a hydrogen bond?","aspirin","hba"),
 ("Report the hydrogen-bond acceptor count for this structure.","ethanol","hba"),
 ("Find the average molecular mass in g/mol.","caffeine","mw"),
 ("Report molecular weight for the given structure.","benzene","mw"),
 ("Estimate the melting point in Celsius.","water","unsupported"),
 ("Calculate both donor and acceptor counts.","glycine","unsupported")]
assert set(q for q,_,_ in DEV).isdisjoint(q for q,_,_ in TEST)
dev_records=[]
for prompt_name in PROMPTS:
    for question,molecule,target in DEV:
        prediction=route(question,prompt_name)["tool"]
        dev_records.append({"prompt":prompt_name,"question":question,"target":target,"prediction":prediction,"correct":prediction==target})
dev_table=pd.DataFrame(dev_records);dev_accuracy=dev_table.groupby("prompt",sort=False).correct.mean()
selected_prompt=max(PROMPTS,key=lambda p:float(dev_accuracy[p]))
print("Development scores:");display(dev_accuracy);print("Frozen selected prompt:",selected_prompt)
def score_agent(result,molecule,target):
    if target=="unsupported":return result["status"]=="abstained"
    if result["status"]!="answered" or result["decision"]["tool"]!=target:return False
    reference=call(target,CATALOGUE[molecule]);ev=result["evidence"]
    return ev["tool"]==target and ev["smiles"]==reference["smiles"] and ev["unit"]==reference["unit"] and abs(ev["value"]-reference["value"])<1e-6
test_records=[];test_traces=[]
for question,molecule,target in TEST:
    result=run_agent(question,molecule,selected_prompt);test_traces.append(result)
    test_records.append({"question":question,"molecule":molecule,"target":target,"selected_tool":result["decision"]["tool"],"status":result["status"],"tool_calls":len(result["trace"]),"routing_correct":result["decision"]["tool"]==target,"end_to_end_correct":score_agent(result,molecule,target)})
test_table=pd.DataFrame(test_records);display(test_table)
assert all(len(t["trace"])<=3 for t in test_traces)
assert all(t["evidence"] is not None for t in test_traces if t["status"]=="answered")
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(dev_accuracy.index,dev_accuracy.values);axes[0].set(ylim=(0,1.05),ylabel="Routing accuracy",title="Development set only: prompt selection")
metrics={"Routing":float(test_table.routing_correct.mean()),"End-to-end":float(test_table.end_to_end_correct.mean())}
axes[1].bar(metrics.keys(),metrics.values(),color="#0d9488");axes[1].set(ylim=(0,1.05),ylabel="Accuracy",title="Held-out wording: 8 teaching cases")
for i,v in enumerate(metrics.values()):axes[1].text(i,v+0.02,f"{v:.0%}",ha="center")
finish(fig,"04_prompt_and_test_results.png")
checks["disjoint_wording_frozen_selection_bounded_traces"]=True


## 8. Save evidence and verify reloads

Save the selected prompt, model identity/revision, environment, raw answer, evaluation cases, and action–observation traces. Do not copy a gigabyte of pretrained weights into the notebook or repository. The model can be reloaded from the pinned public checkpoint/cache; here we verify a tokenizer round trip and a saved prompt's uncached routing result. This is **not** a model-weight checkpoint round trip.

Recomputing each saved descriptor verifies that the recorded number matches RDKit for that stored structure. It cannot detect every intent error or establish that the user's original molecule was correct.


In [ ]:
artifact={"model_id":MODEL_ID,"model_revision":MODEL_REVISION,"selected_prompt":selected_prompt,"prompt_text":PROMPTS[selected_prompt],"labels":LABELS}
(OUT/"agent_config.json").write_text(json.dumps(artifact,indent=2));loaded=json.loads((OUT/"agent_config.json").read_text());assert loaded==artifact
probe_question=TEST[0][0];before=route(probe_question,selected_prompt)
route_cache.pop((probe_question,selected_prompt),None);after=route(probe_question,loaded["selected_prompt"])
assert before["tool"]==after["tool"] and np.allclose(list(before["conditional_scores"].values()),list(after["conditional_scores"].values()),atol=1e-6)
tokenizer.save_pretrained(OUT/"tokenizer")
restored_tokenizer=AutoTokenizer.from_pretrained(OUT/"tokenizer",local_files_only=True,trust_remote_code=False)
assert tokenizer.encode(probe_question)==restored_tokenizer.encode(probe_question)
(OUT/"traces.json").write_text(json.dumps(test_traces,indent=2));saved_traces=json.loads((OUT/"traces.json").read_text());assert saved_traces==test_traces
for trace in saved_traces:
    if trace["status"]=="answered":
        evidence=trace["evidence"];recomputed=call(evidence["tool"],evidence["smiles"])
        assert recomputed["value"]==evidence["value"] and recomputed["unit"]==evidence["unit"]
dev_table.to_csv(OUT/"development.csv",index=False);test_table.to_csv(OUT/"test_results.csv",index=False)
checks["config_tokenizer_trace_reload_and_evidence_replay"]=True
assert all(checks.values())
summary={"environment":versions,"checks":checks,"figures":figures,"direct_answer":direct_answer,"direct_reference":direct_reference,"direct_strict_numeric_match":direct_exact,
         "selected_prompt":selected_prompt,"development_accuracy":{k:float(v) for k,v in dev_accuracy.items()},
         "test_cases":len(TEST),"test_metrics":metrics,"test_records":test_records,"model_routing_forward_calls":model_route_calls,
         "live_demo_status":live_demo["status"],"live_demo_steps":len(live_demo["trace"]),"elapsed_seconds_after_imports":time.perf_counter()-started}
(OUT/"run_summary.json").write_text(json.dumps(summary,indent=2))
print("TUTORIAL_14_COMPLETE");print(json.dumps(summary,indent=2));print("All workflow checks passed; inspect task accuracy separately.")


## Observed Colab CPU result (2026-10-15)

All nine code cells and controller/reload checks completed. The actual Qwen model gave **2** donors for aspirin without tools; RDKit returned **1**. The live agent selected HBD and recovered from a name-as-SMILES error through three recorded tool calls.

Prompt development accuracy was 4/8 for `brief` and 6/8 for `examples`, so `examples` was selected before testing. On held-out wording, routing and end-to-end accuracy were both **4/8 (50%)**. Two paraphrased donor/acceptor questions were misrouted to molecular weight. Both unsupported requests were answered with unrelated descriptors instead of rejected. These failures are retained; no prompt was retuned on those test questions.

The tool calculates its selected descriptor correctly, but the router can select the wrong property. Therefore, `status="answered"` means execution completed, not that the user's request was answered correctly. This notebook teaches that failure distinction; it does not deliver a dependable chemistry assistant. Four figures were inspected. The run took 93.66 seconds after imports, including model loading/download; time varies. GPU and longer-generation mode remain untested.

## Interpretation and exercises

- Inspect every misrouted question, including unsupported requests. Did the model choose a valid but inappropriate tool?
- How much work did the LLM actually perform? Separate routing from deterministic calculation and programmed recovery.
- Why can constrained decoding ensure a valid label but not factual correctness?
- Add a new property by extending the registry, action space, prompt, and evaluation cases together. Keep new test wording separate from prompt development.
- As a separate experiment, let an LLM propose JSON calls and repair them from observations. Keep schema validation and a strict budget; measure invalid calls and recovery success. That open-ended behavior is not tested here.
- Compare prompt choices on a larger independently written dataset, multiple label orders, and another model. Four-way conditional scores are not calibrated confidence.

**Limits:** A small instruction model, two candidate prompts, an embedded six-molecule catalogue, and 16 hand-written questions are teaching tools. Prompt selection is not fine-tuning; the controller is not an autonomous planner; chemical descriptors are not materials performance predictions. No paid API, DSPy/GSM8K, GPU, or longer-generation mode is claimed as verified by the default run.

## Troubleshooting and attribution

If the public model download fails, check connectivity and rerun; do not substitute canned model outputs. After dependency conflicts, start a fresh runtime and run from the top. Download `outputs/tutorial_14/` before Colab disconnects.

License information is available in the repository LICENSE. Additional reference: [LAMA Lab's LLM tutorial](https://github.com/lamalab-org/llm-tutorial). 

References: [MRKL](https://arxiv.org/abs/2205.00445), [ReAct](https://arxiv.org/abs/2210.03629), [DSPy](https://github.com/stanfordnlp/dspy), [Qwen checkpoint and license](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct), [RDKit descriptors](https://www.rdkit.org/docs/GettingStartedInPython.html#list-of-available-descriptors). We record actions, observations, and results rather than requesting hidden chain-of-thought text.
